In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.animation as animation
import time as time
from math import ceil
import re
import os
import matplotlib.patches as patches



from scipy.integrate  import trapezoid, cumulative_trapezoid

import matplotlib.pyplot as plt



import sys
import os

# go one folder up from the current working directory
parent_dir = os.path.abspath("..")

# build the path to folder_b
path = os.path.join(parent_dir)

# add it to sys.path
sys.path.append(path)

import vector_borne_functions as vbf


T = vbf.T
f = vbf.f
gammas = vbf.gammas
g = vbf.g
chis = vbf.chis
F = vbf.F
daily_mean_arrays = vbf.daily_mean_arrays

dX_ms = vbf.dX_ms
Vector_borne_ms = vbf.Vector_borne_ms

dX_Rinf_ms = vbf.dX_Rinf_ms

#Vector_borne_Rinf_ms = vbf.Vector_borne_Rinf_ms

Vector_borne_Rinf_noise = vbf.Vector_borne_Rinf_noise


plot_compartments_vs_t = vbf.plot_compartments_vs_t

In [3]:
#list to store the failed jobs (simulation data that should be in R_folders and do not)
failed_jobs = []


# path to the directory with the data of R_{\infty}}
cwd = os.getcwd()

#parent = os.path.dirname(cwd) + "/adding_noise"

parent_dir = cwd  + '/R_infty_arrays'


# list of folder in the folder "R_0_arrays"
R_folders = os.listdir(parent_dir)
#chosing one folder
#folder_name = input(f'las carpetas que hay son {R_folders} elige una:')
folder_name = R_folders[0]

# path to the folder with data of the simulations with "dimension", "a_min", ...
directory = parent_dir + '/' + folder_name

# names of the arrays in the folder with data of the simulations with "dimension", "a_min", ...
# these arrays have names Arr_{Alpha}_{beta}_{mu}_{Gamma}

folder_arrays = os.listdir(directory)

# parameters of the simulation that give rise to the data in each array
disordered_parameters = [[float(x) for x in re.findall(r"[-+]?\d*\.?\d+", y)] for y in folder_arrays if re.search(r'\d', y)]


disordered_parameters = np.array(disordered_parameters)

ordered_parameters = np.loadtxt(directory + '/params.txt')

parameters = np.array([row for row in ordered_parameters  if  any((row == disordered_parameters).all(axis=1))])

# list to store the arrays 
R_inf_arrays = []


for parameter_combination in parameters:
    
    Alpha, sigma, realization = parameter_combination
    
    R_inf_arrays.append(np.loadtxt(directory + f'/R_inf_values_{Alpha}_{sigma}_{realization}.txt'))


# path to the directory with the data of R_{\infty}}
#cwd = os.getcwd()


sigma_0_folder = cwd  + '/R_infty_arrays' + '/dimension_70_arage_-20_26_brange_12_50_dfraction_5_n200_Iv_noise_sigma_0'

R_inf_sigma_0_arrays = []

parameters_sigma_0 = np.loadtxt(sigma_0_folder + '/params.txt')


for parameter_combination in parameters_sigma_0:

    Alpha, sigma, realization = parameter_combination

    R_inf_sigma_0_arrays.append(np.loadtxt(sigma_0_folder+ f'/R_inf_values_{Alpha}_{sigma}_{realization}.txt'))


n_realizations = 100


mean_arrays = [sum(R_inf_arrays[x*n_realizations:(x+1)*n_realizations])/n_realizations for x in range(9)]

mean_arrays_rs_1 = mean_arrays[:3]
mean_arrays_rs_2 = mean_arrays[3:6]
mean_arrays_rs_3 = mean_arrays[6:9]

mean_delta_rs_1 = [x - R_inf_sigma_0_arrays[0] for x in mean_arrays_rs_1]
mean_delta_rs_2 = [x - R_inf_sigma_0_arrays[1] for x in mean_arrays_rs_2]
mean_delta_rs_3 = [x - R_inf_sigma_0_arrays[2] for x in mean_arrays_rs_3]


np.savetxt('delta_r_inf.txt',mean_delta_rs_1[2][:,1:])